In [1]:
# 导入所需库
import pandas as pd  # 数据处理库
import numpy as np   # 数值计算库
import warnings      # 警告处理库
import re            # 正则表达式库
import datetime      # 日期时间处理库
from difflib import SequenceMatcher  # 字符串匹配库
warnings.filterwarnings("ignore")  # 忽略警告信息

# 定义文件路径
input_path = r"./原始数据-盖锡咨询-中国项目数据库历史数据-202509预测 .xlsx"  # 原始数据路径
output_path1 = r"./预测每月拆分地区集中式202509v2.xlsx"  # 集中式项目输出路径
output_path2 = r"./预测每月拆分地区分布式202509v2.xlsx"  # 分布式项目输出路径
output_path3 = r"./预测每月拆分地区类型202509v2.xlsx"    # 按项目类型输出路径

# 读取原始数据（读取Excel中的"国内中标"工作表）
df_data = pd.read_excel(input_path, sheet_name='国内中标')
df_data.info()  # 查看数据基本信息（列名、数据类型、非空值数量等）

# 筛选集中式项目数据
# 条件1：招标内容为"EPC总承包"或"组件采购"
# 条件2：项目类型属于集中式（如光伏治沙、地面电站等）
df1 = df_data[(df_data['招标内容'] == 'EPC总承包') | (df_data['招标内容'] == '组件采购')]
df1 = df1[(df1['项目类型'] == '光伏治沙') | (df1['项目类型'] == '地面电站')| (df1['项目类型'] == '农光互补')| (df1['项目类型'] == '风光互补')
          | (df1['项目类型'] == '年度采购') | (df1['项目类型'] == '集中采购')| (df1['项目类型'] == '框架采购')
          | (df1['项目类型'] == '平价上网') | (df1['项目类型'] == '领跑者')| (df1['项目类型'] == '年度集采')]
df1['项目规模MW'] = df1['项目规模MW'].fillna(0)  # 项目规模缺失值填充为0

# 筛选分布式项目数据
# 条件1：招标内容为"EPC总承包"或"组件采购"
# 条件2：项目类型为"分布式"或"扶贫"
df2 = df_data[(df_data['招标内容'] == 'EPC总承包') | (df_data['招标内容'] == '组件采购')]
df2 = df2[(df2['项目类型'] == '分布式') | (df2['项目类型'] == '扶贫')]
df2['项目规模MW'] = df2['项目规模MW'].fillna(0)  # 项目规模缺失值填充为0

# 筛选所有符合招标内容的项目（用于按类型统计）
df3 = df_data[(df_data['招标内容'] == 'EPC总承包') | (df_data['招标内容'] == '组件采购')]
df3['项目规模MW'] = df3['项目规模MW'].fillna(0)  # 项目规模缺失值填充为0

# 提取分组维度（用于后续输出表的行索引）
cat1 = df1['项目地址-省']  # 集中式项目按"省份"分组
cat2 = df2['项目地址-省']  # 分布式项目按"省份"分组
cat3 = df3['项目类型']     # 所有项目按"项目类型"分组

# 创建输出表结构（列包含分组维度+时间序列月份）
# 时间范围从2021年1月到2054年11月（部分月份）
df_output1 = pd.DataFrame(data=cat1, columns=['项目地址-省',
                                          '202101','202102','202103','202104','202105','202106',
                                          '202107','202108','202109','202110','202111','202112',
                                          '202201','202202','202203','202204','202205','202206',
                                          '202207','202208','202209','202210','202211','202212',
                                          '202301','202302','202303','202304','202305','202306',
                                          '202307','202308','202309','202310','202311','202312',
                                          '202401','202402','202403','202404','202405','202406',
                                          '202407','202408','202409','202410','202411','202412',
                                          '202501','202502','202503','202504','202505','202506',
                                          '202507','202508','202509','202510','202511','202512',
                                          '202601','202602','202603','202604','202605','202606',
                                          '202607','202608','202609','202610','202611','202612',
                                          '202701','202702','202703','202704','202705','202706',
                                          '202707','202710',
                                          '202801','202803','202804','202805','202806','202811',
                                          '202901','202903','202908',
                                          '203010','203012',
                                          '203101',
                                          '203510','203512',
                                          '203707','203711',
                                          '203911',
                                          '204402','204410',
                                          '204504',
                                          '205006',
                                          '205411'])

# 分布式项目输出表（结构同集中式，仅分组维度为省份）
df_output2 = pd.DataFrame(data=cat2, columns=['项目地址-省',
                                          '202101','202102','202103','202104','202105','202106',
                                          # ... 省略重复的月份列（同df_output1）
                                          '205411'])

# 按项目类型输出表（分组维度为项目类型，月份列同前）
df_output3 = pd.DataFrame(data=cat3, columns=['项目类型',
                                          '202101','202102','202103','202104','202105','202106',
                                          # ... 省略重复的月份列（同df_output1）
                                          '205411'])

print(df_output3["项目类型"].unique())  # 查看所有 unique 的项目类型

# 集中式项目：按月份分摊项目规模
df1['交付月份'] = df1['交付月份'].fillna(0)  # 交付月份缺失值填充为0
for i in df1.index.tolist():  # 遍历每个项目
    # 将中标月份和交付月份转换为字符串（与输出表的列名格式一致）
    df1.loc[i,'中标月份'] = str(int(df1.loc[i,'中标月份']))
    df1.loc[i,'交付月份'] = str(int(df1.loc[i,'交付月份']))
    # 检查中标月份和交付月份是否在输出表的列中（避免索引错误）
    if (df1.loc[i,'中标月份'] in df_output1.columns) and (df1.loc[i,'交付月份'] in df_output1.columns):
        month_win = df1.loc[i,'中标月份']  # 中标月份
        month_fin = df1.loc[i,'交付月份']  # 交付月份
        # 计算每月分摊的项目规模（总规模 / 中标到交付的月数）
        mw_per_month = df1.loc[i,'项目规模MW'] / len(df_output1.loc[i,month_win:month_fin])
        # 将分摊值填充到对应的月份列中
        df_output1.loc[i,month_win:month_fin] = mw_per_month

# 分布式项目：按月份分摊项目规模（逻辑同集中式）
df2['交付月份'] = df2['交付月份'].fillna(0)
for i in df2.index.tolist():
    df2.loc[i,'中标月份'] = str(int(df2.loc[i,'中标月份']))
    df2.loc[i,'交付月份'] = str(int(df2.loc[i,'交付月份']))
    if (df2.loc[i,'中标月份'] in df_output2.columns) and (df2.loc[i,'交付月份'] in df_output2.columns):
        month_win = df2.loc[i,'中标月份']
        month_fin = df2.loc[i,'交付月份']
        mw_per_month = df2.loc[i,'项目规模MW'] / len(df_output2.loc[i,month_win:month_fin])
        df_output2.loc[i,month_win:month_fin] = mw_per_month

# 按项目类型：按月份分摊项目规模（逻辑同前）
df3['交付月份'] = df3['交付月份'].fillna(0)
for i in df3.index.tolist():
    df3.loc[i,'中标月份'] = str(int(df3.loc[i,'中标月份']))
    df3.loc[i,'交付月份'] = str(int(df3.loc[i,'交付月份']))
    if (df3.loc[i,'中标月份'] in df_output3.columns) and (df3.loc[i,'交付月份'] in df_output3.columns):
        month_win = df3.loc[i,'中标月份']
        month_fin = df3.loc[i,'交付月份']
        mw_per_month = df3.loc[i,'项目规模MW'] / len(df_output3.loc[i,month_win:month_fin])
        df_output3.loc[i,month_win:month_fin] = mw_per_month
        print(df_output3.loc[i,month_win:month_fin])  # 打印当前项目的月份分摊结果

# 汇总数据（按分组维度求和）
df_large = df_output1.groupby('项目地址-省').sum()  # 集中式项目按省份汇总
df_small = df_output2.groupby('项目地址-省').sum()  # 分布式项目按省份汇总
df_cat = df_output3.groupby('项目类型').sum()      # 所有项目按类型汇总

# 导出结果到Excel
df_large.to_excel(output_path1)
df_small.to_excel(output_path2)
df_cat.to_excel(output_path3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28204 entries, 0 to 28203
Data columns (total 61 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   招标日期                          25699 non-null  object        
 1   中标日期                          28204 non-null  datetime64[ns]
 2   中标月份                          28204 non-null  int64         
 3   标案状态                          16870 non-null  object        
 4   项目编号                          15872 non-null  object        
 5   项目名称                          28204 non-null  object        
 6   项目类型                          28202 non-null  object        
 7   招标人                           28170 non-null  object        
 8   项目地址-省                        27996 non-null  object        
 9   项目地址-市                        27454 non-null  object        
 10  中标人                           28145 non-null  object        
 11  中标公司简称                      